# 03 — Bird Data

Loads real-world bird observation data, calibrates model parameters, and compares bird-derived dynamics with the baseline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.graph_objects as go

from data.loader import DataLoader
from data.calibration import (
    scale_initial_count, scale_reproduce_rate, scale_gain_from_food,
    build_ecosystem_config_from_birds,
)
from simulation.runner import SimulationRunner, RunConfig
from model.config import EcosystemConfig

loader = DataLoader()
runner = SimulationRunner()
print('Setup complete.')

## 1. Load Sample Data

In [ ]:
# Generate sample data if it doesn't exist
sample_path = '../data/sample/sample_bird_data.csv'
loader.generate_sample_data(sample_path)

df = loader.load_csv(sample_path)
display(df)

# Show calibration output for each row
print('\nCalibration outputs:')
for _, row in df.iterrows():
    gain = scale_gain_from_food(float(row['territory_km2']), row['role'])
    rep  = scale_reproduce_rate(int(row['breeding_pairs']), int(row['observed_count']))
    cnt  = scale_initial_count(int(row['observed_count']), 2500)
    print(f"  {row['species']:16s}  count={cnt:3d}  reproduce={rep:.4f}  gain_from_food={gain}")

## 2. Build Config from Birds

In [ ]:
species_configs = loader.to_species_configs(df)
bird_cfg = build_ecosystem_config_from_birds(species_configs)

print('Resulting EcosystemConfig:')
print(f'  initial_wolves        = {bird_cfg.initial_wolves}  (Sparrowhawk count, log-scaled from 35,000)')
print(f'  initial_sheep         = {bird_cfg.initial_sheep}  (average of Blue Tit + House Sparrow counts)')
print(f'  wolf_reproduce        = {bird_cfg.wolf_reproduce:.4f}  (breeding_pairs/observed_count ÷ 52)')
print(f'  sheep_reproduce       = {bird_cfg.sheep_reproduce:.4f}  (average of prey reproduce rates)')
print(f'  wolf_gain_from_food   = {bird_cfg.wolf_gain_from_food}  (inverse log of Sparrowhawk territory 6 km²)')
print(f'  sheep_gain_from_food  = {bird_cfg.sheep_gain_from_food}  (average of prey gain_from_food)')
print(f'  initial_wolf_energy   = {bird_cfg.initial_wolf_energy}  (3 × gain_from_food)')
print(f'  initial_sheep_energy  = {bird_cfg.initial_sheep_energy}  (3 × gain_from_food)')

### Assumptions and how to override them

Each calibration step encodes an ecological assumption:

| Parameter | Assumption | Override if... |
|---|---|---|
| `initial_count` | Log-scale from observed count; cap at 30% grid density | You have density/area data for the study site |
| `reproduce_rate` | Annual breeding ratio ÷ 52 weeks | Species breeds seasonally or has multi-year cycles |
| `gain_from_food` | Inverse-log of territory size | You have direct calorific data from literature |
| `initial_energy` | 3× gain_from_food | You want agents to survive longer before first feeding |

## 3. Run Bird-Derived Model vs Baseline

In [ ]:
baseline_cfg = EcosystemConfig()
baseline_rc  = RunConfig(ecosystem_config=baseline_cfg, n_steps=300, seed=42, stop_on_extinction=False)
baseline     = runner.run(baseline_rc)

bird_rc  = RunConfig(ecosystem_config=bird_cfg, n_steps=300, seed=42, stop_on_extinction=False)
bird_run = runner.run(bird_rc)

print('Baseline  steps:', baseline.steps_completed, '| extinct:', baseline.extinction_events or 'None')
print('Bird-derived steps:', bird_run.steps_completed, '| extinct:', bird_run.extinction_events or 'None')

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Baseline Parameters', 'Bird-Derived Parameters'],
    shared_yaxes=False)

for col, (res, label) in enumerate([(baseline, 'Baseline'), (bird_run, 'Bird-derived')], start=1):
    d = res.data
    fig.add_trace(go.Scatter(x=d.index, y=d['Wolves'], name=f'{label} Wolves',
        line=dict(color='red', dash='solid' if col==1 else 'dot')), row=1, col=col)
    fig.add_trace(go.Scatter(x=d.index, y=d['Sheep'], name=f'{label} Sheep',
        line=dict(color='steelblue', dash='solid' if col==1 else 'dot')), row=1, col=col)

fig.update_layout(title='Baseline vs Bird-Derived Population Dynamics', hovermode='x unified')
fig.show()

## 4. Manual Calibration Walkthrough

In [ ]:
import dataclasses

# Override wolf_gain_from_food based on published Sparrowhawk calorific data
# Sparrowhawks consume ~25g prey per hunt; Blue Tit ~18g → rough energy ratio ~1.4
# We can encode this as a manual override:
manual_cfg = dataclasses.replace(bird_cfg, wolf_gain_from_food=15)

manual_rc  = RunConfig(ecosystem_config=manual_cfg, n_steps=300, seed=42, stop_on_extinction=False)
manual_run = runner.run(manual_rc)

print(f'Manual override: wolf_gain_from_food=15 (was {bird_cfg.wolf_gain_from_food})')
print(f'Steps: {manual_run.steps_completed} | Extinct: {manual_run.extinction_events or "None"}')

### When is manual override scientifically appropriate?

Manual overrides are appropriate when:

1. **Published data contradicts the scaling heuristic** — e.g. a species has an unusually high conversion efficiency for its territory size.
2. **The study site is atypical** — urban gardens have higher prey density than the national average implied by territory_km2.
3. **Temporal mismatch** — breeding data is from a different year than the population count.

Document every override in a config comment or README so the calibration chain remains reproducible.

### Data sources for UK birds

- **BTO BirdFacts**: https://www.bto.org/our-science/data — breeding population estimates, territory sizes, survival rates
- **RSPB Species Explorer**: https://www.rspb.org.uk/birds-and-wildlife/wildlife-guides/bird-a-z/ — accessible species profiles with habitat and diet
- **GBIF Occurrence API**: https://www.gbif.org/developer/summary — georeferenced occurrence records, suitable for density mapping
- **BirdTrack**: https://www.bto.org/our-science/projects/birdtrack — migration and seasonal count data